# 27. 사업보고서 검색 방식 비교

**목적:** 26번의 제목 경계 우선 청킹(B)을 고정하고, 어휘 검색·의미 벡터 검색·두 순위를 결합한 하이브리드 검색 중 사업보고서 근거를 더 잘 찾는 방식을 비교합니다.

**고정 조건:** 삼성전자·리노공업의 2024 사업보고서 각 1건, 25번 B 원문 추출, `SentenceSplitter(512, overlap=64)` 제목 경계 청킹, 회사별 검색, 같은 정답 근거와 질문. 26번의 개발 질문 10개에 실제 리서치 표현 6개를 추가합니다. 추가 질문은 **같은 근거를 다시 묻는 재표현**이므로 독립 holdout이 아닙니다.

**후보:** (1) 26번과 같은 글자 2~4-gram `HashingVectorizer`, (2) 로컬 `multilingual-e5-small` dense embedding, (3) 둘의 순위를 RRF로 결합한 hybrid. 이 노트북은 BM25·reranker·LLM 답변·최종 Tool을 평가하지 않습니다.

**해석:** Hit@3/5와 MRR@5를 기본·실제형 재표현 그룹별로 보며, 정답 인용문이 청크에 없으면 검색 방식 비교를 중단합니다. 모델 로드·인덱스 생성·검색 시간을 분리해 기록합니다. 작은 개발셋이므로 여기서 최종 운영 검색기를 확정하지 않습니다.

## 1. 환경과 평가 질문

반드시 `skn25` 커널에서 실행합니다. E5 모델은 첫 실행 때 Hugging Face에서 약 471 MB의 가중치를 내려받을 수 있고, 이 실험에서는 CPU만 사용해 이미 GPU에 올라간 Qwen에 영향을 주지 않습니다. 외부 LLM API는 호출하지 않습니다.

In [ ]:
from io import BytesIO
from pathlib import Path
from time import perf_counter
from importlib.metadata import version
import os, re, zipfile

from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from dotenv import load_dotenv
from IPython.display import display
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from sklearn.feature_extraction.text import HashingVectorizer
from transformers import AutoModel, AutoTokenizer
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / '.env').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 최상위 .env를 찾지 못했습니다.')
load_dotenv(PROJECT_ROOT / '.env')
API_KEY = os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('.env에 OPENDART_API_KEY가 필요합니다.')
print('환경 | llama-index-core', version('llama-index-core'), '| transformers', version('transformers'), '| torch', version('torch'))

In [ ]:
REPORTS = {
    '삼성전자': {'rcept_no': '20250311001085', 'xml_name': '20250311001085.xml'},
    '리노공업': {'rcept_no': '20250318000229', 'xml_name': '20250318000229.xml'},
}
GOLD = [
    ('S01', '삼성전자', '삼성전자의 2024년 시설투자 규모는 얼마였나?', '53.6조원의 시설투자가 이루어졌습니다.'),
    ('S02', '삼성전자', '삼성전자가 관리하는 주요 재무 위험에는 무엇이 있나?', '시장위험, 신용위험, 유동성위험 등을 최소화하는데 중점을 두고'),
    ('S03', '삼성전자', '삼성전자 그룹에 포함된 종속기업 규모는?', 'SDC, Harman 산하 종속기업 등 228개의 종속기업으로 구성된 글로벌 전자 기업입니다.'),
    ('S04', '삼성전자', '삼성전자 TV 사업의 2024년까지 판매 1위 지속 기간은?', 'TV사업은 2006년부터 2024년까지 19년 연속 판매 1위를 지키고 있습니다.'),
    ('S05', '삼성전자', '삼성전자 모바일사업의 글로벌 출하량 1위 지속 기간은?', '모바일사업은 2011년부터 2024년까지 14년 연속 글로벌 출하량 1위를 유지하고 있습니다.'),
    ('R01', '리노공업', '리노공업은 제품별 가격을 산출하기 어려운 이유가 무엇인가?', '30,000여종의 제품을 생산하고 있어, 특정 제품에 대한 가격을 개별적으로 산출하기는 어렵습니다.'),
    ('R02', '리노공업', '리노공업은 제품 공급가격을 정할 때 무엇을 고려하는가?', '거래처별 공급수량 및 제품의 난이도에 따라 합리적으로 가격을 책정'),
    ('R03', '리노공업', '리노공업이 주시하는 시장위험 요인에는 무엇이 있나?', '가. 시장위험과 위험관리 국내외 원자재 가격과 유가의 불안정, 경기 변동, 지정학적 위험'),
    ('R04', '리노공업', '리노공업 연구개발의 기본 개념은 무엇인가?', '전 사원의 연구원화'),
    ('R05', '리노공업', '리노공업은 해외 고객을 어떻게 관리하는가?', '지역별 혹은 국가별로 현지 대리점을 두어 고객을 관리하고 있으며'),
]
PRACTICAL = [
    ('P01', 'S01', '삼성전자가 2024년 설비투자에 얼마를 썼는지 공시 근거로 확인해줘.'),
    ('P02', 'S02', '삼성전자 사업보고서에서 투자자가 주의해 볼 재무상 위험의 종류를 정리해줘.'),
    ('P03', 'S03', '삼성전자 연결대상 자회사는 몇 곳인가? 보고서 근거를 보여줘.'),
    ('P04', 'R01', '리노공업은 제품별 판매가격을 한 값으로 비교하기 어려운 이유가 뭐야?'),
    ('P05', 'R03', '리노공업이 사업보고서에서 언급한 외부 시장 위험은 무엇인가?'),
    ('P06', 'R05', '리노공업은 해외 고객을 어떤 방식으로 관리하고 있어?'),
]
by_id = {row[0]: row for row in GOLD}
CASES = [{'id': i, 'group': '기본', 'company': company, 'question': question, 'quote': quote}
         for i, company, question, quote in GOLD]
CASES += [{'id': i, 'group': '실제형 재표현', 'company': by_id[source][1], 'question': question, 'quote': by_id[source][3]}
          for i, source, question in PRACTICAL]
display(pd.DataFrame(CASES)[['id', 'group', 'company', 'question']])

## 2. 26번과 같은 사업보고서 텍스트·B 청크 준비

OpenDART에서 같은 접수번호의 원문 ZIP을 다시 받고, 같은 XML·같은 제목 경계 규칙을 사용합니다. 인증키와 원문 전체는 출력하지 않습니다.

In [ ]:
def download_xml(rcept_no, xml_name):
    try:
        with requests.get('https://opendart.fss.or.kr/api/document.xml',
                          params={'crtfc_key': API_KEY, 'rcept_no': rcept_no},
                          timeout=120, stream=True) as response:
            if response.status_code != 200:
                raise RuntimeError(f'OpenDART 원문 HTTP {response.status_code}')
            parts, received = [], 0
            for part in response.iter_content(chunk_size=1024 * 1024):
                received += len(part)
                if received > 60_000_000:
                    raise RuntimeError('원문 ZIP이 60MB를 넘어 다운로드를 중단합니다.')
                parts.append(part)
    except requests.RequestException:
        raise RuntimeError('OpenDART 원문 요청 실패: 연결·인증·응답 상태를 확인하세요. 요청 URL은 출력하지 않습니다.') from None
    payload = b''.join(parts)
    if not zipfile.is_zipfile(BytesIO(payload)):
        raise RuntimeError('OpenDART 응답이 ZIP 형식이 아닙니다. 키·접수번호·API 상태를 확인하세요.')
    try:
        with zipfile.ZipFile(BytesIO(payload)) as archive:
            if xml_name not in archive.namelist():
                raise RuntimeError('26번에서 선택한 XML 파일을 ZIP에서 찾지 못했습니다.')
            info = archive.getinfo(xml_name)
            if info.file_size > 60_000_000 or sum(item.file_size for item in archive.infolist()) > 120_000_000:
                raise RuntimeError('ZIP 압축 해제 크기가 실험 한도를 넘습니다.')
            return archive.read(xml_name)
    except zipfile.BadZipFile:
        raise RuntimeError('OpenDART 원문 ZIP이 손상됐습니다. 접수번호와 응답 형식을 확인하세요.') from None

In [ ]:
def extract_b_text(xml_bytes):
    soup = BeautifulSoup(xml_bytes, 'lxml-xml')
    if soup.find() is None:
        raise RuntimeError('XML에서 태그를 찾지 못했습니다.')
    parts = []
    for node in soup.descendants:
        if isinstance(node, Tag) and node.name.upper() == 'TR':
            parts.append('\n')
        if not isinstance(node, NavigableString) or isinstance(node, Comment):
            continue
        value = re.sub(r'\s+', ' ', str(node)).strip()
        if not value:
            continue
        ancestors = {tag.name.upper() for tag in node.parents if tag.name}
        if ancestors & {'TITLE', 'SUBTITLE'}:
            parts.append('\n## ' + value + '\n')
        elif ancestors & {'TD', 'TH'}:
            parts.append(value + ' | ')
        else:
            parts.append(value + ' ')
    return re.sub(r' *\n *', '\n', ''.join(parts)).strip()

In [ ]:
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=64)
chunks_by_company, build_rows = {}, []
for company, report in REPORTS.items():
    started = perf_counter()
    text = extract_b_text(download_xml(report['rcept_no'], report['xml_name']))
    sections = [part for part in re.split(r'(?=^## )', text, flags=re.MULTILINE) if part.strip()]
    chunks = [node.text.strip() for section in sections
              for node in splitter.get_nodes_from_documents([Document(text=section)], show_progress=False)
              if node.text.strip()]
    chunks_by_company[company] = chunks
    build_rows.append({'company': company, 'rcept_no': report['rcept_no'],
                       'sections': len(sections), 'chunks': len(chunks),
                       'download_extract_chunk_seconds': round(perf_counter() - started, 2)})
display(pd.DataFrame(build_rows))
assert [len(chunks_by_company[c]) for c in REPORTS] == [96, 384], '26번과 청크 수가 달라졌습니다. 원문·버전·분할 규칙을 확인하세요.'
compact = lambda value: re.sub(r'\s+', '', value)
if any(not any(compact(case['quote']) in compact(chunk) for chunk in chunks_by_company[case['company']]) for case in CASES):
    raise RuntimeError('정답 인용문이 B 청크에 없습니다. 검색 평가를 중단합니다.')
print('근거 검증 완료 | 16개 질문의 인용문이 각 회사 청크에 적어도 1회 포함됨')

## 3. 어휘 검색 인덱스

26번과 동일한 문자 2~4-gram 검색기입니다. 이름에 ‘벡터’가 있어도 학습된 의미 임베딩은 아닙니다.

In [ ]:
vectorizer = HashingVectorizer(analyzer='char', ngram_range=(2, 4),
                               n_features=2**18, norm='l2', alternate_sign=False, dtype=np.float32)
lexical_index, lexical_rows = {}, []
for company, chunks in chunks_by_company.items():
    started = perf_counter()
    matrix = vectorizer.fit_transform(chunks)
    lexical_index[company] = matrix
    matrix_bytes = matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
    lexical_rows.append({'company': company, 'index_seconds': round(perf_counter() - started, 2),
                         'sparse_MB': round(matrix_bytes / 1024**2, 2)})
display(pd.DataFrame(lexical_rows))

## 4. 로컬 의미 임베딩 모델 로드

`multilingual-e5-small`은 한국어를 포함하는 다국어 모델이며 모델 카드가 `query: `·`passage: ` 접두어와 평균 풀링을 안내합니다. 첫 다운로드와 로드 시간은 검색 지연시간과 분리합니다. GPU에 이미 올린 Qwen을 건드리지 않도록 CPU로 고정합니다. 이 모델의 512-token 입력 한계를 초과한 청크 수를 반드시 확인합니다.

In [ ]:
MODEL_ID = 'intfloat/multilingual-e5-small'
MODEL_REVISION = '5697a65b0a002a92fe8c4fc9d495303ffff9c7d2'
torch.set_num_threads(min(4, os.cpu_count() or 1))
print('모델 로드 시작 | 첫 실행에서는 약 471 MB 다운로드가 필요할 수 있습니다.')
started = perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_safetensors=True).to('cpu').eval()
model_load_seconds = perf_counter() - started
model_parameter_MB = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
print(f'모델 로드 완료 | {model_load_seconds:.1f}초 | 파라미터 메모리 약 {model_parameter_MB:.0f} MiB | CPU threads {torch.get_num_threads()}')

In [ ]:
def encode_texts(texts, kind, batch_size=16):
    encoded_rows = []
    for start in range(0, len(texts), batch_size):
        batch = [f'{kind}: {text}' for text in texts[start:start + batch_size]]
        tokens = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt')
        with torch.inference_mode():
            hidden = model(**tokens).last_hidden_state
            mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            normalized = F.normalize(pooled, p=2, dim=1)
        encoded_rows.append(normalized.cpu().numpy().astype(np.float32))
    return np.vstack(encoded_rows)

length_rows = []
for company, chunks in chunks_by_company.items():
    lengths = [len(tokenizer('passage: ' + chunk, add_special_tokens=True, truncation=False)['input_ids']) for chunk in chunks]
    length_rows.append({'company': company, 'over_512': sum(length > 512 for length in lengths),
                        'max_tokens': max(lengths)})
display(pd.DataFrame(length_rows))
if any(row['over_512'] for row in length_rows):
    print('주의: 일부 청크가 E5 512-token 한계를 넘어 뒤쪽이 잘립니다. dense/hybrid 결과의 최종 채택을 보류하세요.')

In [ ]:
dense_index, dense_rows = {}, []
for company, chunks in chunks_by_company.items():
    print(f'{company} passage 임베딩 시작 | {len(chunks)}개, CPU 실행')
    started = perf_counter()
    matrix = encode_texts(chunks, 'passage')
    dense_index[company] = matrix
    dense_rows.append({'company': company, 'index_seconds': round(perf_counter() - started, 2),
                       'shape': str(matrix.shape), 'dense_MB': round(matrix.nbytes / 1024**2, 2)})
display(pd.DataFrame(dense_rows))

## 5. 세 검색기의 순위와 근거 위치 비교

어휘 점수와 dense 점수는 크기를 직접 더하지 않고, 각 검색기의 **순위**를 RRF로 합칩니다. 각 검색기의 상위 20개만 fusion 후보로 넣으며 `k=60`을 고정합니다. 모델·청킹·질문은 세 방식에서 같습니다.

In [ ]:
def rrf_order(lex_order, dense_order, size, depth=20, k=60):
    scores = np.zeros(size, dtype=np.float64)
    for order in (lex_order, dense_order):
        ids = order[:depth]
        scores[ids] += 1.0 / (k + np.arange(1, len(ids) + 1))
    order = np.argsort(-scores, kind='stable')
    return order[scores[order] > 0]

def first_evidence_rank(order, chunks, quote):
    needle = compact(quote)
    return next((rank for rank, idx in enumerate(order, 1) if needle in compact(chunks[idx])), None)

assert list(rrf_order(np.array([0, 1, 2]), np.array([0, 2, 1]), 3)) == [0, 1, 2]
assert first_evidence_rank([1, 0], ['정답 문장', '오답'], '정답문장') == 2
print('검색 순위·근거 판정 최소 검사 완료')

In [ ]:
rows = []
for case in CASES:
    company, question = case['company'], case['question']
    chunks = chunks_by_company[company]
    started = perf_counter()
    lex_query = vectorizer.transform([question])
    lex_scores = (lexical_index[company] @ lex_query.T).toarray().ravel()
    lex_order = np.argsort(-lex_scores, kind='stable')
    lex_ms = (perf_counter() - started) * 1000
    started = perf_counter()
    dense_query = encode_texts([question], 'query')[0]
    dense_scores = dense_index[company] @ dense_query
    dense_order = np.argsort(-dense_scores, kind='stable')
    dense_ms = (perf_counter() - started) * 1000
    started = perf_counter()
    hybrid_order = rrf_order(lex_order, dense_order, len(chunks))
    hybrid_ms = lex_ms + dense_ms + (perf_counter() - started) * 1000
    for method, order, latency in [('어휘', lex_order, lex_ms), ('벡터', dense_order, dense_ms),
                                   ('하이브리드', hybrid_order, hybrid_ms)]:
        rank = first_evidence_rank(order, chunks, case['quote'])
        rows.append({'id': case['id'], 'group': case['group'], 'company': company, 'method': method,
                     'rank': rank, 'hit_at_3': rank is not None and rank <= 3,
                     'hit_at_5': rank is not None and rank <= 5,
                     'rr_at_5': 1 / rank if rank is not None and rank <= 5 else 0.0,
                     'query_ms': round(latency, 2),
                     'status': 'RRF 양쪽 상위 20개 밖' if method == '하이브리드' and rank is None else '검색 순위 있음',
                     'top1_excerpt': chunks[order[0]][:160]})
    print(f"[{case['id']}] 어휘 {rows[-3]['rank']}위 | 벡터 {rows[-2]['rank']}위 | 하이브리드 {rows[-1]['rank']}위")
results = pd.DataFrame(rows)
display(results[['id', 'group', 'method', 'rank', 'status', 'hit_at_3', 'hit_at_5', 'query_ms']])

In [ ]:
summary = results.groupby(['group', 'method'], as_index=False).agg(
    cases=('id', 'count'), hit_at_3=('hit_at_3', 'mean'),
    hit_at_5=('hit_at_5', 'mean'), mrr_at_5=('rr_at_5', 'mean'),
    median_query_ms=('query_ms', 'median'))
display(summary)
misses = results[~results['hit_at_5']]
print('상위 5개에서 근거를 놓친 사례:', len(misses))
display(misses[['id', 'group', 'method', 'rank', 'status', 'top1_excerpt']])

## 결과를 읽는 기준

1. 각 그룹에서 Hit@3/5·MRR@5와 실패 질문을 함께 봅니다. `P01~P06`은 기존 근거를 다시 묻는 질문이므로 16개를 독립 표본처럼 합산해 최종 우열을 선언하지 않습니다.
2. 시설투자 질문 `S01/P01`의 순위가 개선되는지 확인합니다. 이 질문은 26번에서 근거가 존재해도 상위 5개에서 놓친 사례입니다.
3. `over_512 > 0`이면 dense 모델이 일부 청크의 뒤쪽을 보지 못합니다. 그 경우 embedding·hybrid 결과를 최종 채택 근거로 사용하지 않습니다.
4. 어휘 arm은 26번의 문자 n-gram이며 **BM25가 아닙니다**. 이 노트북은 검색 구조의 첫 기준선입니다. 하이브리드가 반드시 최고라는 가정은 하지 않습니다.
5. 모델 로드·passage 임베딩 생성은 최초 준비 비용이며 `query_ms`에 포함되지 않습니다. `query_ms`는 CPU에서 질문 1개를 검색하는 warm 실행 시간입니다. 표본 수가 적어 운영 지연시간을 대표하지 않습니다.
6. 다음 parent–child 비교에서는 선택한 검색 방식을 고정하고, 상위 검색 순위뿐 아니라 **전달되는 문맥의 정답 포함 여부·토큰 사용량**도 비교해야 합니다.

참고: [multilingual-E5-small 모델 카드](https://huggingface.co/intfloat/multilingual-e5-small), [LlamaIndex 순위 결합 예제](https://developers.llamaindex.ai/python/examples/low_level/fusion_retriever/), [scikit-learn HashingVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.HashingVectorizer.html), [LlamaIndex Retrieval Evaluation](https://developers.llamaindex.ai/python/framework/understanding/evaluating/evaluating/).